In [1]:
import re
import numpy as np
from pathlib import Path
from typing import Tuple
import json
import math

In [210]:
TARGET_NAME="LeNet-300-100"
TARGET_DIR="../results/1.LeNet-300-100"
TARGET_DATE="20260407"
TARGET_TYPE="canonical"
MODEL_ID1=1
MODEL_ID2=3
TARGET_CONFIG=f"freq_{MODEL_ID1}_{MODEL_ID2}.json"

In [2]:
def parse_c_header(file_path: str) -> Tuple[str, np.ndarray]:
    """
    Parse C header file and extract array name and data

    Args:
        file_path: Path to the C header file

    Returns:
        Tuple of (array_name, data_array)
    """
    with open(file_path, "r") as f:
        content = f.read()

    # Find array declaration
    array_pattern = r"unsigned\s+char\s+([a-zA-Z0-9_]+)\s*\[\]\s*=\s*\{"
    match = re.search(array_pattern, content)

    if not match:
        raise ValueError(f"Could not find array declaration in {file_path}")

    array_name = match.group(1)

    # Extract hex values
    hex_pattern = r"0x([0-9a-fA-F]{2})"
    hex_values = re.findall(hex_pattern, content)

    if not hex_values:
        raise ValueError(f"Could not find hex values in {file_path}")

    # Convert to numpy array
    data = np.array([int(h, 16) for h in hex_values], dtype=np.uint8)

    return array_name, data

In [4]:
# CONFIG_PATH = Path(f"{TARGET_DIR}/{TARGET_DATE}/{TARGET_TYPE}/{TARGET_CONFIG}")

with open("../results/2.LeNet-5/models/config_delta_LeNet-5_9_10.json") as f:
    cfg = json.load(f)

best = cfg["best_encoding"]
HUFFMAN_CODES = best["huffman_codes"]  # symbol(str) → bitstring
MERGED_VALUES = set(best["merged_values"])
REPRESENTATIVE_VALUE = best["representative_value"] # special rare-delta token
REPRESENTATIVE_CODE = HUFFMAN_CODES[str(REPRESENTATIVE_VALUE)]
BITSLENGTH = best["bitsLen_symNum"]
MIN_BITS_LENGTH = best["min_code_length"]
MAX_BITS_LENGTH = best["max_code_length"]

# Build reverse map  bitstring → int symbol  (for decoding)
REVERSE_CODES = {v: int(k) for k, v in HUFFMAN_CODES.items()}

print(f"[config]  total symbols in Huffman table : {len(HUFFMAN_CODES)}")
print(f"[config]  merged (rare) values           : {len(MERGED_VALUES)}")
print(f"[config]  representative code for {REPRESENTATIVE_VALUE}   : {REPRESENTATIVE_CODE}")
avg_bits = cfg["best_avg_bits"]
print(
    f"[config]  best avg bits/symbol           : {avg_bits:.4f}  "
    f"(vs 8 raw bits → {avg_bits/8*100:.1f}% of original)"
)

[config]  total symbols in Huffman table : 126
[config]  merged (rare) values           : 386
[config]  representative code for 48   : 110100
[config]  best avg bits/symbol           : 5.4027  (vs 8 raw bits → 67.5% of original)


In [5]:
# with open(
#     f"{TARGET_DIR}/q-hufdel_{TARGET_NAME}_{MODEL_ID1}_to_{MODEL_ID2}.bin", "rb"
# ) as f:
#     package = f.read()
with open(
    "../results/2.LeNet-5/models/Q-HUFDEL_LeNet-5_9_10.bin",
    "rb",
) as f:
    package = f.read()

print(type(package))
print(len(package))  # size in bytes

<class 'bytes'>
49040


In [6]:
for i in range(0, 80, 8):
    chunk = package[i:i+8]
    print(' '.join(f'{b:02X}' for b in chunk))

51 2D 48 55 46 44 45 4C
01 00 09 00 0A 00 7E 00
03 0C 08 09 30 00 00 00
35 B0 A2 23 F0 36 5B B0
9F 3F 08 48 71 9C C0 7D
78 1A 01 00 1A F6 05 00
01 02 0E 09 07 07 1F 11
0E 18 00 40 40 30 88 3C
1A 0B 04 81 C0 80 40 30
20 14 0C 07 04 43 21 70


In [12]:
name1, data1 = parse_c_header("../results/2.LeNet-5/models/LeNet-5_9-quantized.h")
name2, data2 = parse_c_header("../results/2.LeNet-5/models/LeNet-5_10-quantized.h")

print(name1, "\n", name2)
print(f"Data 1 shape: {data1.shape}, Data 2 shape: {data2.shape}")

___models_2_LeNet_5_quantized_LeNet_5_9_quantized_tflite 
 ___models_2_LeNet_5_quantized_LeNet_5_10_quantized_tflite
Data 1 shape: (72312,), Data 2 shape: (72312,)


In [194]:
name1, data1 = parse_c_header(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID1}-quantized.h")
name2, data2 = parse_c_header(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID2}-quantized.h")

print(name1, "\n", name2)
print(f"Data 1 shape: {data1.shape}, Data 2 shape: {data2.shape}")

___models_1_LeNet_300_100_quantized_LeNet_300_100_1_quantized_tflite 
 ___models_1_LeNet_300_100_quantized_LeNet_300_100_3_quantized_tflite
Data 1 shape: (270720,), Data 2 shape: (270720,)


In [13]:
deltas = data2.astype(np.int16) - data1.astype(np.int16)
assert len(deltas) == len(
    data2.flat
), f"Length mismatch: delta={len(deltas)}, model2={len(data2.flat)}"

# Decode 
## Header

In [14]:
import struct
import zlib
from dataclasses import dataclass

MAGIC = b"Q-HUFDEL"  # exactly 8 bytes — frame 1


# Data class representing the decoded header
@dataclass
class HufDelHeader:
    # Frame 2
    version_type: int  # V-T  (1 byte)
    update_type: int  # U-T  (1 byte)
    version_current: int  # current version of dest file  (2 bytes, LE)
    version_update: int  # update version of dest file   (2 bytes, LE)
    num_huffman_symbols: int  # number of Huffman symbols (2 byte LE)

    # Frame 3
    min_code_len: int  # min code length of symbol      (1 byte)
    max_code_len: int  # max code length of symbol      (1 byte)
    replaced_bits: int  # replaced value bit-width, e.g. 8 (1 byte)
    symbol_bits: int  # symbol value bit-width (1 byte)
    special_symbol: int  # symbol with special handling (e.g. merged values) (2 byte)
    # ↑ 2 bytes used — 2 bytes reserved padding to fill frame 3

    # Frame 4: checksum placeholder (0xFF * 8) — algorithm MD5
    checksum: bytes  # 16-byte checksum (MD5) stored as two 8-byte LE frames

    # Frame 5
    target_size_bytes: int  # size of model2 in bytes        (4 bytes, LE)
    bitstream_size_bits: int  # total bitstream payload bytes  (4 bytes, LE)


# Encoder
HEADER_FRAME_COUNT = 6
HEADER_SIZE = HEADER_FRAME_COUNT * 8  # 48 bytes

In [15]:
# Decoder
def decode_header(data: bytes) -> tuple[HufDelHeader, bytes]:
    """
    Parse the 40-byte header from the front of a package.

    Returns:
        (HufDelHeader, remainder)  — remainder is everything after the header.

    Raises:
        ValueError  on magic mismatch or truncated data.
    """
    if len(data) < HEADER_SIZE:
        raise ValueError(
            f"Package too short: expected at least {HEADER_SIZE} bytes, "
            f"got {len(data)}."
        )

    # ── Frame 1: magic check ──────────────────────────────────────────────────
    magic = data[0:8]
    if magic != MAGIC:
        raise ValueError(f"Magic mismatch: expected {MAGIC!r}, got {magic!r}")

    # ── Frame 2: version / type flags ────────────────────────────────────────
    (
        version_type,
        update_type,
        version_current,
        version_update,
        num_huffman_symbols,
    ) = struct.unpack_from("<BBHHH", data, offset=8)

    # ── Frame 3: code / bit-width metadata ───────────────────────────────────
    (
        min_code_len,
        max_code_len,
        replaced_bits,
        symbol_bits,
        special_symbol,
    ) = struct.unpack_from("<BBBBh", data, offset=16)

    # ── Frame 4: checksum placeholder
    (f1_le, f2_le) = struct.unpack_from("<QQ", data, offset=24)

    # Convert to big endian (reverse bytes)
    f1_be = struct.pack(">Q", f1_le)
    f2_be = struct.pack(">Q", f2_le)
    checksum = f1_be + f2_be

    # ── Frame 5: sizes ────────────────────────────────────────────────────────
    (
        target_size_bytes,
        bitstream_size_bits,
    ) = struct.unpack_from("<II", data, offset=40)

    header = HufDelHeader(
        version_type=version_type,
        update_type=update_type,
        version_current=version_current,
        version_update=version_update,
        num_huffman_symbols=num_huffman_symbols,
        min_code_len=min_code_len,
        max_code_len=max_code_len,
        replaced_bits=replaced_bits,
        symbol_bits=symbol_bits,
        special_symbol=special_symbol,
        checksum=checksum,
        target_size_bytes=target_size_bytes,
        bitstream_size_bits=bitstream_size_bits,
    )

    remainder = data[HEADER_SIZE:]
    return header, remainder

In [16]:
header, remainder = decode_header(package)
print(header)
print(f"Remainder size: {len(remainder)} bytes")

assert header.num_huffman_symbols == len(
    HUFFMAN_CODES
), f"Header num_huffman_symbols {header.num_huffman_symbols} does not match config {len(HUFFMAN_CODES)}"

assert (
    header.min_code_len == MIN_BITS_LENGTH
), f"Header min_code_len {header.min_code_len} does not match config {MIN_BITS_LENGTH}"

assert (
    header.max_code_len == MAX_BITS_LENGTH
), f"Header max_code_len {header.max_code_len} does not match config {MAX_BITS_LENGTH}"

assert (
    header.special_symbol == REPRESENTATIVE_VALUE
), f"Header special_symbol {header.special_symbol} does not match config {REPRESENTATIVE_VALUE}"

# import hashlib

# with open(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID2}-quantized.h", "rb") as f:
#     md5_hex = hashlib.md5(f.read()).hexdigest()
# assert (
#     md5_hex == header.checksum.hex()
# ), f"Checksum mismatch: expected {header.checksum.hex()}, got {md5_hex}"
# print(f"Checksum verified: {md5_hex}")

assert header.target_size_bytes == len(
    data2
), f"Header target_size_bytes {header.target_size_bytes} does not match actual {len(data2)}"

HufDelHeader(version_type=1, update_type=0, version_current=9, version_update=10, num_huffman_symbols=126, min_code_len=3, max_code_len=12, replaced_bits=8, symbol_bits=9, special_symbol=48, checksum=b'\xb0[6\xf0#\xa2\xb05}\xc0\x9cqH\x08?\x9f', target_size_bytes=72312, bitstream_size_bits=390682)
Remainder size: 48992 bytes


## Symbol Lookup Table

In [17]:
def symbol_section_size(
    bitslen_symnum: dict[str, int],
    huffman_codes:  dict[str, str],
) -> int:
    """Return the byte size of the encoded symbol section (always a multiple of 8)."""
    n_counts  = len(bitslen_symnum)
    n_symbols = len(huffman_codes)
    total_bits = n_counts * 8 + n_symbols * 9
    return math.ceil(total_bits / 64) * 8   # round up to full 8-byte frames

# ─────────────────────────────────────────────────────────────────────────────
# 9-bit sign-magnitude helpers
# ─────────────────────────────────────────────────────────────────────────────

def encode_sym9(value: int) -> str:
    """
    Encode a symbol value (-255..255) as a 9-bit sign-magnitude bitstring.
        bit[0]   : sign  (0 = positive / zero, 1 = negative)
        bits[1-8]: magnitude as unsigned 8-bit
    e.g.  -7  →  1_00000111  →  "100000111"
          7   →  0_00000111  →  "000000111"
          0   →  0_00000000  →  "000000000"
    """
    if not (-255 <= value <= 255):
        raise ValueError(f"Symbol value {value} out of range -255..255")
    sign = 1 if value < 0 else 0
    mag  = abs(value)
    return f"{sign}{mag:08b}"


def decode_sym9(bits9: str) -> int:
    """
    Decode a 9-bit sign-magnitude bitstring back to an integer.
    """
    sign = int(bits9[0])
    mag  = int(bits9[1:], 2)
    return -mag if sign else mag


def symbol_section_size_from_counts(n_lengths: int, n_symbols: int) -> int:
    """Byte size of the section given raw counts (no dicts needed)."""
    total_bits = n_lengths * 8 + n_symbols * 9
    return math.ceil(total_bits / 64) * 8

In [18]:
def decode_symbol_table(
    data:        bytes,
    n_lengths:   int,     # number of distinct code lengths  = max_code - min_code + 1
    n_symbols:   int,     # total number of Huffman symbols  = num_huffman_symbols
    min_code_len:int,     # from header — to reconstruct the length keys
) -> tuple[dict[int, int], bytes]:
    """
    Decode the unified symbol section from the front of `data`.

    Returns:
        (symbol_code_len, remainder)

        symbol_code_len : dict  int_symbol_value → code_length
        remainder       : bytes immediately after the symbol section
                          (i.e. the start of the bitstream payload)
    """

    # ── How many bytes does this section occupy? ──────────────────────────────
    section_bytes = symbol_section_size_from_counts(n_lengths, n_symbols)

    if len(data) < section_bytes:
        raise ValueError(
            f"Symbol section too short: need {section_bytes} bytes, "
            f"got {len(data)}."
        )

    # ── Unpack section bytes → full bitstring (drop padding implicitly) ───────
    raw      = data[:section_bytes]
    bitstring = "".join(f"{b:08b}" for b in raw)

    pos = 0

    # ── Section A: read n_lengths counts (8 bits each) ───────────────────────
    counts: dict[int, int] = {}
    for i in range(n_lengths):
        counts[min_code_len + i] = int(bitstring[pos : pos + 8], 2)
        pos += 8

    # ── Section B: read n_symbols values (9 bits each) ───────────────────────
    symbols: list[int] = []
    for _ in range(n_symbols):
        symbols.append(decode_sym9(bitstring[pos : pos + 9]))
        pos += 9

    # ── Reconstruct symbol → code_length mapping ─────────────────────────────
    symbol_code_len: dict[int, int] = {}
    sym_idx = 0
    for code_len in sorted(counts.keys()):
        count = counts[code_len]
        for sym in symbols[sym_idx : sym_idx + count]:
            symbol_code_len[sym] = code_len
        sym_idx += count

    remainder = data[section_bytes:]
    return symbol_code_len, remainder


In [19]:
symbol_code_len, remainder = decode_symbol_table(
    data=remainder,
    n_lengths=(header.max_code_len - header.min_code_len + 1),  # from header
    n_symbols=header.num_huffman_symbols,  # from header
    min_code_len=header.min_code_len,  # from header
)

In [20]:
assert len(symbol_code_len) == header.num_huffman_symbols, "Number of symbols mismatch"

In [21]:
def build_canonical_codebook(lengths):
    """
    Derive the canonical Huffman codes from a symbol→length mapping.

    Canonical Huffman rules
    ───────────────────────
    1. Codes of shorter length come *numerically before* codes of longer length.
    2. Within the same length, codes are assigned in *lexicographic order* of
       the symbols (alphabetical for characters).
    3. The first code of length L is determined by left-shifting the previous
       length's last code and adding 1:

           code[0 of new length] = (code[-1 of prev length] + 1) << (new_L - prev_L)

    This produces a compact, self-describing table: a decoder only needs to
    store the symbol list sorted by (length, symbol) and the starting code for
    each length — not the full tree.

    Returns {symbol: bit-string}, e.g. {'a': '0', 'b': '10', 'c': '11'}.
    """
    if not lengths:
        return {}

    # Sort symbols: first by code-length, then alphabetically (canonical order)
    sorted_symbols = sorted(lengths.items(), key=lambda kv: (kv[1], kv[0]))

    codebook: dict[int, str] = {}
    current_code = 0
    prev_length = 0

    for symbol, length in sorted_symbols:
        # Shift left by the increase in bit-length to maintain prefix-free property
        current_code <<= length - prev_length
        # Represent the integer as a zero-padded binary string of the right width
        codebook[symbol] = format(current_code, f"0{length}b")
        current_code += 1  # next symbol at same length gets the next integer
        prev_length = length

    return codebook

In [22]:
codebook = build_canonical_codebook(symbol_code_len)

## Payload

In [23]:
def decode_payload(
    payload: bytes,
    model1_flat,  # model1 weights, flat, same order as encoder
    reverse_codes,  # bitstring → int symbol
):
    """
    Decode a Huffman-compressed package back into model2 weights.

    Payload layout (n * 8 bytes total):
        [ payload bits ... | pad bits ] [ CRC-32 : 4 bytes ]

    CRC is skipped per spec. pad_bits must be stored externally by the encoder.
    """

    # ── 1. Strip CRC (last 4 bytes), keep payload
    payload_without_crc = payload[:-4]

    # ── 2. Bytes → bitstring, strip padding
    bitstring = "".join(f"{byte:08b}" for byte in payload_without_crc)

    # ── 3. Build prefix trie for O(1)-per-bit decoding
    # Walk the bitstring one bit at a time, accumulating until we match a code.
    weights = []
    huffman_hits = 0
    fallback_hits = 0

    pos = 0
    total = len(bitstring)
    m1_idx = 0  # index into model1_flat

    while pos < total:
        # ── 3a. Read bits until a known code is matched
        buf = ""
        symbol = None

        while pos < total:
            buf += bitstring[pos]
            pos += 1

            if buf in reverse_codes:
                symbol = reverse_codes[buf]
                break

            # Guard: no valid code can be longer than the longest code in table
            if len(buf) > max(len(k) for k in reverse_codes):
                raise ValueError(
                    f"No matching Huffman code found at bit position {pos}, "
                    f"buffer={buf!r}"
                )

        if symbol is None:
            break  # trailing pad bits exhausted — done

        # ── 3b. Dispatch on symbol
        if buf == REPRESENTATIVE_CODE:
            # Next 8 bits = raw unsigned uint8 weight (no model1 involved)
            if pos + 8 > total:
                raise ValueError(f"Fallback at bit {pos} but fewer than 8 bits remain.")
            raw_bits = bitstring[pos : pos + 8]
            pos += 8
            weight = int(raw_bits, 2)  # uint8, 0–255
            weights.append(weight)
            fallback_hits += 1
        else:
            # Normal delta: weight = model1[i] + decoded_delta (clamped to uint8)
            delta = symbol
            base = int(model1_flat[m1_idx])
            weight = (base + delta) & 0xFF  # keep as uint8
            weights.append(weight)
            huffman_hits += 1

        m1_idx += 1

        # Stop once we've recovered as many weights as model1 has
        if m1_idx >= len(model1_flat):
            break

    return weights, huffman_hits, fallback_hits

In [24]:
weights, huffman_hits, fallback_hits = decode_payload(
    payload=remainder,  # the n*8 bytes from the encoder
    model1_flat=list(data1.flat),
    reverse_codes=REVERSE_CODES,
)

# Sanity check
assert huffman_hits + fallback_hits == len(
    data1.flat
), f"Decoded {len(weights)} weights, but model1 has {len(data1.flat)}"

reconstructed = np.array(weights, dtype=np.uint8).reshape(data2.shape)

In [25]:
if np.array_equal(data2, reconstructed):
    print("success")
else:
    print("fail")

success
